
# 練習問題: モンテカルロ法で円周率 π を推定する (各スレッドが独立に)

## 目標

work-sharing 構文 (`for` / `do`) や `reduction` はまだ学んでいない. ここでは `omp_get_thread_num()` と `omp_get_num_threads()` だけを使って, モンテカルロ法による π の推定を複数スレッドで行う. 各スレッドは**自分の担当分の点を投げ, 自分自身の π 推定値を表示する**. 共有変数への足し込み (集約) はまだ行わない.

## 背景: モンテカルロ法

単位正方形 `[0,1) x [0,1)` 内に一様乱数で点を投げると, 原点中心・半径 1 の 1/4 円 (`x^2 + y^2 < 1`) の内側に落ちる確率は, その面積比 `π/4` に等しい. したがって

```
(円内に落ちた点の数) / (投げた点の総数) ≒ π/4
```

なので, この比を 4 倍すれば π の推定値が得られる. 点を多く投げるほど π ≒ 3.14159 に近づく.

## 課題

`montecarlo.cpp` (または `montecarlo.f90`) は, 全体で `N` 個 (コマンドライン引数, 既定 4,000,000) の点を投げる. 各スレッド `t` (全 `T` スレッド) は `N/T` 個の点を投げ, 円内に落ちた数を数えて自分の π 推定値を表示する.

コメント `TODO` の指示に従って **OpenMP の指示行を追加** し, このブロックを複数スレッドで実行させよ.

- C++: ブロック `{ ... }` の直前に `#pragma omp parallel` を1行加える. スレッドごとの変数 (`tid`, `hits`, 乱数種 `seed` など) はブロック内で宣言してあるので, 自動的にスレッドごとに別々になる.
- Fortran: ブロックを `!$omp parallel private(...)` と `!$omp end parallel` で囲む. スレッドごとに別々にしたい変数を `private` 節に並べる.

**注意:** 1つの共有変数に全スレッドの `hits` を足し込んではならない (それは競合 (data race) になる. 総和をまとめる `reduction` は後のトピック). 各スレッドは**自分の**推定値だけを表示すること.

乱数は, スレッドごとに独立・スレッド安全である必要がある. C++ では種をスレッド番号 + 1 とした `rand_r` を, Fortran では各スレッドが独立した状態を持つ自前の線形合同法 (LCG) を用いている. これらは既に書かれているので変更不要.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore montecarlo.cpp -o montecarlo.exe

# Fortran
nvfortran -fast -mp=multicore montecarlo.f90 -o montecarlo.exe
```

```
OMP_NUM_THREADS=4 ./montecarlo.exe
OMP_NUM_THREADS=4 ./montecarlo.exe 40000000
```

## 期待される結果

`OMP_NUM_THREADS` に指定した数だけ行が表示され, 各スレッドが自分の推定値を出す (順番は実行ごとに入れ替わってよい). 各推定値は π ≒ 3.14159 に近い値になる. 例 (4スレッド):

```
thread 0 of 4: 1000000 points, pi estimate = 3.141234
thread 1 of 4: 1000000 points, pi estimate = 3.140876
thread 2 of 4: 1000000 points, pi estimate = 3.142560
thread 3 of 4: 1000000 points, pi estimate = 3.141100
```

投げる点の総数 `N` を増やすほど, 各推定値が π に近づくことを確認せよ. (穴あきのまま実行すると 1 スレッドだけが動き, 行が1つしか出ない.)



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:montecarlo.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:montecarlo.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ montecarlo.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

int main(int argc, char **argv) {
  // 全体で投げる点の数 (コマンドライン引数, 既定 4,000,000)
  long N = (argc > 1) ? atol(argv[1]) : 4000000L;
  // TODO: 下のブロックの直前に #pragma omp parallel を1行追加し, 各スレッドが自分の担当分の点を投げて, 自分の π 推定値を表示するようにせよ.
  {
    int tid = omp_get_thread_num();
    int nt  = omp_get_num_threads();
    // このスレッドの担当する点数 (N を T スレッドで分割)
    long lo = tid * N / nt;
    long hi = (tid + 1) * N / nt;
    long my_n = hi - lo;
    // スレッドごとに異なる乱数種 (rand_r はスレッド安全)
    unsigned int seed = tid + 1;
    long hits = 0;
    for (long i = 0; i < my_n; i++) {
      double x = rand_r(&seed) / (double)RAND_MAX;
      double y = rand_r(&seed) / (double)RAND_MAX;
      if (x * x + y * y < 1.0) {
        hits++;
      }
    }
    // 単位正方形に対する 1/4 円の面積比 = π/4
    double pi = (my_n > 0) ? 4.0 * hits / my_n : 0.0;
    printf("thread %d of %d: %ld points, pi estimate = %f\n",
           tid, nt, my_n, pi);
  }
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore montecarlo.cpp -o montecarlo_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./montecarlo_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./montecarlo_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:montecarlo.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ montecarlo.f90
program montecarlo
  use omp_lib
  implicit none
  integer(8) :: n, lo, hi, my_n, hits, i
  integer :: tid, nt
  integer(8) :: state
  real(8) :: x, y, pi
  character(len=32) :: arg

  ! 全体で投げる点の数 (コマンドライン引数, 既定 4,000,000)
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg)
     read(arg, *) n
  else
     n = 4000000_8
  end if

  ! TODO: 下のブロックを !$omp parallel private(tid, nt, lo, hi, my_n, hits, i, state, x, y, pi) ... !$omp end parallel で囲み, 各スレッドが自分の担当分の点を投げて自分の π 推定値を表示するようにせよ.
  tid = omp_get_thread_num()
  nt  = omp_get_num_threads()
  ! このスレッドの担当する点数 (n を T スレッドで分割)
  lo = tid * n / nt
  hi = (tid + 1) * n / nt
  my_n = hi - lo
  ! スレッドごとに異なる乱数種 (各スレッドが独立した LCG を持つ)
  state = int(tid + 1, 8)
  hits = 0
  do i = 1, my_n
     x = lcg(state)
     y = lcg(state)
     if (x * x + y * y < 1.0d0) then
        hits = hits + 1
     end if
  end do
  ! 単位正方形に対する 1/4 円の面積比 = π/4
  if (my_n > 0) then
     pi = 4.0d0 * real(hits, 8) / real(my_n, 8)
  else
     pi = 0.0d0
  end if
  print "(a,i0,a,i0,a,i0,a,f0.6)", &
       "thread ", tid, " of ", nt, ": ", my_n, " points, pi estimate = ", pi
  ! TODO: 上で始めた parallel 領域を閉じる (!$omp end parallel).

contains

  ! 単純な線形合同法 (LCG). state を更新し [0,1) の乱数を返す.
  ! Fortran 組込みの乱数はスレッド安全とは限らないので自前で実装する.
  function lcg(s) result(r)
    integer(8), intent(inout) :: s
    real(8) :: r
    ! 2^31 を法とする LCG (Numerical Recipes 系の係数)
    s = mod(1103515245_8 * s + 12345_8, 2147483648_8)
    r = real(s, 8) / 2147483648.0d0
  end function lcg

end program montecarlo

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore montecarlo.f90 -o montecarlo_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./montecarlo_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./montecarlo_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:montecarlo.f90}